## Instalando pacotes necessários

In [ ]:
#!uv pip install duckdb pandas
#!uv pip install python-dotenv
#!uv pip install boto3

## Importando as bibliotecas necessárias

In [ ]:
import duckdb
import pandas as pd
import os
import boto3
from dotenv import load_dotenv
from datetime import datetime
from botocore.client import Config
from botocore.exceptions import NoCredentialsError, EndpointConnectionError

load_dotenv()


#from dotenv import load_dotenv

True

## Busca informacoes no .env

In [22]:
ACCESS_KEY = os.getenv('MINIO_SENHA')
SECRET_KEY = os.getenv('MINIO_LOGIN')
MINIO_URL = os.getenv('MINIO_URL')

## Monta a conexao com o MinIO

In [9]:
def get_s3_client():
    return boto3.client('s3',
        endpoint_url=MINIO_URL,
        aws_access_key_id=ACCESS_KEY,
        aws_secret_access_key=SECRET_KEY,
        config=Config(signature_version='s3v4'),
        region_name='us-east-1'
    )


In [16]:
def testar_conexao():
    try:
        s3 = get_s3_client()
        # Tenta listar os buckets
        response = s3.list_buckets()
        print("Conexão estabelecida com sucesso!")
        print("Buckets existentes:", [b['Name'] for b in response['Buckets']])
    except EndpointConnectionError:
        print("Erro: Não foi possível conectar ao endpoint. Verifique a MINIO_URL.")
    except Exception as e:
        print(f"Erro na conexão: {e}")

testar_conexao()

Conexão estabelecida com sucesso!
Buckets existentes: ['bronze', 'ouro', 'prata']


## Testa a conexao do DuckDB com o MinIO

In [27]:
BRONZE_BUCKET = 'bronze/dados_relacionais/FAC2FTER/areas_de_interesse'
DUCKDB_FILE = 'datalake_analytics.duckdb'

def init_duckdb():
    print(f"🦆 Conectando ao DuckDB: {DUCKDB_FILE}")
    con = duckdb.connect(DUCKDB_FILE)
    
    return con


In [28]:
def test_duckdb_s3_connection(con):
    try:
        # 1. Limpeza do Endpoint (DuckDB espera apenas o host:porta)
        # Remove 'http://' ou 'https://' se existirem
        clean_endpoint = MINIO_URL.replace("http://", "").replace("https://", "")

        con.execute("INSTALL httpfs; LOAD httpfs;")
        con.execute(f"SET s3_endpoint='{clean_endpoint}';")
        con.execute(f"SET s3_access_key_id='{ACCESS_KEY}';")
        con.execute(f"SET s3_secret_access_key='{SECRET_KEY}';")
        con.execute("SET s3_use_ssl=false;") 
        con.execute("SET s3_url_style='path';")

        print(f"🔍 Testando acesso ao bucket: s3://{BRONZE_BUCKET}/")
        
        # 2. FORMA CORRETA DE LISTAR: Usando a função glob do DuckDB
        # Isso substitui o comando 'LS' que falhou
        sql_check = f"SELECT * FROM glob('s3://{BRONZE_BUCKET}/*')"
        result = con.execute(sql_check).fetchall()
        
        print(f"✅ Conectado! Encontrados {len(result)} arquivos no bucket '{BRONZE_BUCKET}'.")
        
    except Exception as e:
        print(f"❌ Erro detalhado: {type(e).__name__}")
        print(f"📝 Mensagem: {e}")

# Execução
con = init_duckdb()
test_duckdb_s3_connection(con)

🦆 Conectando ao DuckDB: datalake_analytics.duckdb
🔍 Testando acesso ao bucket: s3://bronze/dados_relacionais/FAC2FTER/areas_de_interesse/
✅ Conectado! Encontrados 1 arquivos no bucket 'bronze/dados_relacionais/FAC2FTER/areas_de_interesse'.
